# 掼蛋·扑克牌识别模型训练（Colab 一键版）

在 **Google Colab 免费 GPU** 上训练识别扑克牌「数字+花色」的模型(YOLO)。
练完得到模型文件，接到掼蛋 App：摄像头识别 → 自动组牌 → 出牌建议。

**用法**：菜单「运行时 → 更改运行时类型 → 选 GPU」，再「运行时 → 全部运行」。


## 1. 确认已分到 GPU


In [ ]:
!nvidia-smi

## 2. 安装训练框架（Ultralytics YOLO）


In [ ]:
!pip -q install ultralytics roboflow
import ultralytics; ultralytics.checks()

## 3. 获取扑克牌数据集 ⚠️必须改这格

在 https://universe.roboflow.com 选数据集 → Download Dataset → 格式 YOLOv8 →
Show download code，把它给的**整段代码**复制替换下面这格再运行。


In [ ]:
# ↓↓↓ 用 Roboflow『Show download code』整段替换这里 ↓↓↓
from roboflow import Roboflow
rf = Roboflow(api_key="把你的API_KEY粘到这里")
project = rf.workspace("augmented-startups").project("playing-cards-ow27d")
dataset = project.version(4).download("yolov8")
# ↑↑↑ 用真实代码替换以上几行 ↑↑↑
print('数据集下载到:', dataset.location)

## 4. 定位数据集（必要时自动解压）并开始训练

有些数据集下下来只有 roboflow.zip 没解压；这格会自动解压并只在数据集目录里找 data.yaml。


In [ ]:
import glob, os, zipfile
from ultralytics import YOLO

root = dataset.location  # 第3格下载到的目录
if not glob.glob(os.path.join(root, '**/data.yaml'), recursive=True):
    zips = glob.glob(os.path.join(root, '*.zip'))
    assert zips, f'{root} 里没有 data.yaml 也没有 zip，内容: {os.listdir(root)}'
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(root)
    print('已解压:', zips[0])

yamls = glob.glob(os.path.join(root, '**/data.yaml'), recursive=True)
assert yamls, f'解压后仍没找到 data.yaml，内容: {os.listdir(root)}'
DATA = yamls[0]
print('使用数据集配置:', DATA)

model = YOLO('yolov8n.pt')   # 要更准可换 yolov8s.pt / yolov8m.pt
model.train(data=DATA, epochs=50, imgsz=640, batch=16,
            patience=20, project='guandan_cards', name='exp')

## 5. 看效果（验证集指标）


In [ ]:
metrics = model.val()
print('mAP50:', metrics.box.map50, ' mAP50-95:', metrics.box.map)

## 6. 导出模型（给 App 用）


In [ ]:
best = 'guandan_cards/exp/weights/best.pt'
m = YOLO(best)
m.export(format='onnx')
print('模型：', best, ' 和 同目录 best.onnx')

## 7. 下载模型到本地


In [ ]:
from google.colab import files
files.download('guandan_cards/exp/weights/best.pt')
files.download('guandan_cards/exp/weights/best.onnx')

## 8. 拿回模型后怎么用

把 `best.pt` 放到本仓库 `vision/`，运行 `python vision/recognize.py 照片.jpg`，
会输出识别到的牌并调用掼蛋引擎给『自动组牌 + 出牌建议』。手机/眼镜端用 `best.onnx`。
